# This notebook generates the final results plots shown in our manuscript

It expects the results of each individual experiment to have already been generated

## Phospho ROC curve

In [ ]:
from collections import Counter
from tqdm import tqdm 
import numpy as np
import pandas as pd
import pickle


import json
from pathlib import Path

import torch
import ppx
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.metrics import auc, roc_curve, balanced_accuracy_score, f1_score

from lightning.pytorch.loggers import CSVLogger
from torchmetrics.classification import BinaryAUROC

def compute_overall_roc_curve(true_labels_dict, predicted_probs_dict):
    """
    Compute the ROC curve and AUC score across all datasets.

    Parameters:
    true_labels_dict (dict): Dictionary where keys are dataset names and values are binary labels (numpy arrays).
    predicted_probs_dict (dict): Dictionary where keys are dataset names and values are predicted probabilities (numpy arrays).

    Returns:
    fpr (numpy array): False positive rates.
    tpr (numpy array): True positive rates.
    roc_auc (float): Area Under the Curve (AUC) score.
    """
    # Flatten all labels and predictions into single lists
    all_labels = np.concatenate([true_labels_dict[ds] for ds in predicted_probs_dict.keys()])
    all_probs = np.concatenate([predicted_probs_dict[ds] for ds in predicted_probs_dict.keys()])

    # Compute ROC curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)

    return fpr, tpr

In [3]:
print("Read labels")
with open("phospho/results/1/PXD012174_phospho_labels.pkl", "rb") as f:
    phospho_labels = pickle.load(f)

print("Read predictions")
with open("phospho/results/1/PXD012174_casanovo_predictions.pkl", "rb") as f:
    cn_predictions = pickle.load(f)

with open("phospho/results/1/PXD012174_binned_baseline_predictions.pkl", "rb") as f:
    binned_predictions = pickle.load(f)

with open("phospho/results/1/PXD012174_depthcharge_predictions.pkl", "rb") as f:
    dc_predictions = pickle.load(f)

with open(f"phospho/results/1/PXD012174_casanovo_predictions_multi.pkl", "rb") as f:
    cn_predictions_multi = pickle.load(f)

Read labels
Read predictions
Read predictions
Read predictions
Read predictions


In [4]:
casanovo_fpr, casanovo_tpr = compute_overall_roc_curve(phospho_labels, cn_predictions)
baseline_fpr, baseline_tpr = compute_overall_roc_curve(phospho_labels, binned_predictions)
pred_fpr, pred_tpr = compute_overall_roc_curve(phospho_labels, dc_predictions)
casanovo_fpr_multi, casanovo_tpr_multi = compute_overall_roc_curve(phospho_labels, cn_predictions_multi)


In [ ]:
# Increase figure resolution
plt.figure(figsize=(3, 3), dpi=300)

# Custom colors for clarity
colors = {
    "casanovo_non_pretrained": "red",
    "binned_embeddings": "blue",
    "casanovo": "green",
    "casanovo_multi": "purple",
    "casanovo_prec": "pink",
    "random": "gray"
}

plt.plot(casanovo_fpr, casanovo_tpr, label=f"Casanovo Foundation, AUC: {auc(casanovo_fpr, casanovo_tpr):.3f}", color=colors["casanovo"], lw=2)
plt.plot(casanovo_fpr_multi, casanovo_tpr_multi, label=f"Casanovo \n(multi-task training), AUC: {auc(casanovo_fpr_multi, casanovo_tpr_multi):.3f}", color=colors["casanovo_multi"], lw=2)
plt.plot(pred_fpr, pred_tpr, where="pre", label=f"End-to-end transformer , AUC: {auc(pred_fpr, pred_tpr):.3f}", color=colors["casanovo_non_pretrained"], lw=2)
plt.plot(baseline_fpr, baseline_tpr, label=f"Binned embeddings, AUC: {auc(baseline_fpr, baseline_tpr):.3f}", color=colors["binned_embeddings"], lw=2)

# Random classifier baseline
plt.plot([0, 1], [0, 1], color=colors["random"], lw=2, linestyle='--')

# Set axis labels with larger font
plt.xlabel("False Positive Rate", fontsize=10)
plt.ylabel("True Positive Rate", fontsize=10)

# Increase font size for legend
plt.legend(loc="lower right", fontsize=6)

# Despine (remove top and right borders)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("ptm_roc.pdf", format="pdf", bbox_inches="tight")

# Show plot
plt.show()

## Learning curve

In [ ]:
results_path = 'phospho/results/'

sizes = []
baseline_aucs = []
dc_aucs = []
cn_aucs = []
for size in [1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1]:
    print(size)

    with open(f"{results_path}{size}/small_labels.pkl", "rb") as f:
        phospho_labels_small = pickle.load(f)
    
    with open(f"{results_path}{size}/PXD012174_binned_baseline_prediction.pkl", "rb") as f:
        binned_predictions_small = pickle.load(f)

    with open(f"{results_path}{size}/PXD012174_depthcharge_predictions.pkl", "rb") as f:
        dc_predictions_small = pickle.load(f)

    with open(f"{results_path}{size}/PXD012174_casanovo_predictions.pkl", "rb") as f:
        cn_predictions_small = pickle.load(f)
        
    fp,tp = compute_overall_roc_curve(phospho_labels_small, binned_predictions_small)
    baseline_aucs.append(auc(fp, tp))
    fp,tp = compute_overall_roc_curve(phospho_labels_small, dc_predictions_small)
    dc_aucs.append(auc(fp, tp))
    fp,tp = compute_overall_roc_curve(phospho_labels_small, cn_predictions_small)
    cn_aucs.append(auc(fp, tp))

    sizes.append(len(tp))

plt.figure(figsize=(3, 3), dpi=300)

plt.plot(sizes, cn_aucs, label='Casanovo foundation',  marker = '.', color=colors['casanovo'], linewidth=2)
plt.plot(sizes, dc_aucs, label='End-to-end transformer',  marker = '.', color=colors['casanovo_non_pretrained'], linewidth=2)
plt.plot(sizes, baseline_aucs,label='Binned embeddings',  marker = '.', color=colors['binned_embeddings'], linewidth=2)

plt.gca().set_xscale('log')
plt.legend()
plt.xlim([5000, 10000000])
plt.xlabel('Number of training spectra', fontsize=10)
plt.ylabel('AUROC', fontsize=10)

# Increase font size for legend
plt.legend(loc="lower right", fontsize=6)
# Despine (remove top and right borders)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("phospho_learning_curve.pdf", format="pdf", bbox_inches="tight")
plt.show()


## Spectrum quality ROC curve

In [ ]:
# Spectrum quality
import matplotlib.pyplot as plt

with open("quality/roc_fpr_tpr.pkl", "rb") as f:
     results = pickle.load(f)

pred_fpr, pred_tpr = results['depthcharge']
baseline_fpr, baseline_tpr = results['binned']
casanovo_fpr, casanovo_tpr = results['casanovo']
casanovo_combined_fpr, casanovo_combined_tpr = results['casanovo_multi']

print(len(casanovo_fpr))

# Increase figure resolution
plt.figure(figsize=(3, 3), dpi=300)

plt.plot(casanovo_fpr, casanovo_tpr, label=f"Casanovo foundation, AUC: {auc(casanovo_fpr, casanovo_tpr):.3f}", 
         color=colors["casanovo"], lw=2)

plt.plot(casanovo_combined_fpr, casanovo_combined_tpr, label=f"Casanovo foundation \n(multi-task training), AUC: {auc(casanovo_combined_fpr, casanovo_combined_tpr):.3f}", 
         color='purple', lw=2)

plt.step(pred_fpr, pred_tpr, where="pre", label=f"End-to-end transformer, AUC: {auc(pred_fpr, pred_tpr):.3f}", 
         color=colors["casanovo_non_pretrained"], lw=2)

plt.plot(baseline_fpr, baseline_tpr, label=f"Binned embeddings, AUC: {auc(baseline_fpr, baseline_tpr):.3f}",
         color=colors["binned_embeddings"], lw=2)

# Random classifier baseline
plt.plot([0, 1], [0, 1], color=colors["random"], lw=2, linestyle='--')

# Set axis labels with larger font
plt.xlabel("False Positive Rate", fontsize=10)
plt.ylabel("True Positive Rate", fontsize=10)

# Increase font size for legend
plt.legend(loc="lower right", fontsize=6)

# Despine (remove top and right borders)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("quality_roc.pdf", format="pdf", bbox_inches="tight")

# Show plot
plt.show()

## Chimericity ROC curve

In [ ]:
# Spectrum chimericity

with open("chimericity/roc_fpr_tpr.pkl", "rb") as f:
     results = pickle.load(f)

pred_fpr, pred_tpr = results['depthcharge']
baseline_fpr, baseline_tpr = results['binned']
casanovo_fpr, casanovo_tpr = results['casanovo']
casanovo_combined_fpr, casanovo_combined_tpr = results['casanovo_multi']

# Increase figure resolution
plt.figure(figsize=(3, 3), dpi=300)

# Custom colors for clarity
colors = {
    "casanovo_non_pretrained": "red",
    "binned_embeddings": "blue",
    "casanovo": "green",
    "random": "gray"
}

plt.plot(casanovo_fpr, casanovo_tpr, label=f"Casanovo foundation, AUC: {auc(casanovo_fpr, casanovo_tpr):.3f}", 
         color=colors["casanovo"], lw=2)

plt.plot(casanovo_combined_fpr, casanovo_combined_tpr, label=f"Casanovo foundation \n(multi-task training), AUC: {auc(casanovo_combined_fpr, casanovo_combined_tpr):.3f}", 
         color='purple', lw=2)

plt.plot(pred_fpr, pred_tpr, label=f"End-to-end transformer, AUC: {auc(pred_fpr, pred_tpr):.3f}", 
         color=colors["casanovo_non_pretrained"], lw=2)

plt.plot(baseline_fpr, baseline_tpr, label=f"Binned embeddings, AUC: {auc(baseline_fpr, baseline_tpr):.3f}",
         color=colors["binned_embeddings"], lw=2)

# Random classifier baseline
plt.plot([0, 1], [0, 1], color=colors["random"], lw=2, linestyle='--')

# Set axis labels with larger font
plt.xlabel("False Positive Rate", fontsize=10)
plt.ylabel("True Positive Rate", fontsize=10)

# Increase font size for legend
plt.legend(loc="lower right", fontsize=6)

# Despine (remove top and right borders)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("chimericity_roc.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Glyco ROC and PR curve

In [ ]:
# PTM
import sklearn
import pickle
import matplotlib.pyplot as plt

# Increase figure resolution
plt.figure(figsize=(3, 3), dpi=300)

with open("glyco/glyco_roc_fpr_tpr.pkl", "rb") as f:
     results = pickle.load(f)

fpr, tpr = results['ratio']
baseline_fpr, baseline_tpr = results['GlyCounter']
binned_fpr, binned_tpr = results['binned']
transformer_fpr, transformer_tpr = results['transformer']
casanovo_fpr, casanovo_tpr = results['casanovo']
casanovo_joint_fpr, casanovo_joint_tpr = results['casanovo_joint']

plt.plot(casanovo_fpr, casanovo_tpr, label=f"Casanovo foundation, AUC:{round(sklearn.metrics.auc(casanovo_fpr, casanovo_tpr), 3)}", color=colors['casanovo'], lw=2)
plt.plot(casanovo_joint_fpr, casanovo_joint_tpr, label=f"Casanovo foundation \n(multi-task training), AUC:{round(sklearn.metrics.auc(casanovo_joint_fpr, casanovo_joint_tpr), 3)}", color='purple', lw=2)
plt.plot(transformer_fpr, transformer_tpr, label=f"End-to-end Transformer, AUC:{round(sklearn.metrics.auc(transformer_fpr, transformer_tpr), 3)}", color=colors["casanovo_non_pretrained"], lw=2)
plt.plot(binned_fpr, binned_tpr, label=f"Binned embeddings, AUC:{round(sklearn.metrics.auc(binned_fpr, binned_tpr), 3)}0", lw=2, color=colors['binned_embeddings'])
plt.plot(baseline_fpr, baseline_tpr, label=f"GlyCounter + XGBoost, AUC:{round(sklearn.metrics.auc(baseline_fpr, baseline_tpr), 3)}", color='orange', lw=2)
plt.plot(fpr, tpr, label=f"138/144 Ratio baseline, AUC:{round(sklearn.metrics.auc(fpr, tpr), 3)}", color='teal', lw=2)

# Random classifier baseline
plt.plot([0, 1], [0, 1], color=colors["random"], lw=2, linestyle='--')

# Set axis labels with larger font
plt.xlabel("False Positive Rate", fontsize=10)
plt.ylabel("True Positive Rate", fontsize=10)

# Increase font size for legend
plt.legend(loc="lower right", fontsize=6)

# Despine (remove top and right borders)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("glyco_roc.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
def convert_to_pr(fpr, tpr):
    num_neg = 42005
    num_pos = 4340
    ps = []
    rs = []
    for fp, tp in zip(fpr[1:], tpr[1:]):
        ps.append(tp*num_pos/(tp*num_pos + fp*num_neg))
        rs.append(tp*num_pos/(num_pos))
    return rs, ps

# Increase figure resolution
plt.figure(figsize=(5, 3), dpi=300)

plt.plot(*convert_to_pr(casanovo_fpr, casanovo_tpr), label=f"Casanovo foundation, AUC:{round(sklearn.metrics.auc(*convert_to_pr(casanovo_fpr, casanovo_tpr)), 3)}", color=colors['casanovo'], lw=2)
plt.plot(*convert_to_pr(casanovo_joint_fpr, casanovo_joint_tpr), label=f"Casanovo foundation \n(multi-task training), AUC:{round(sklearn.metrics.auc(*convert_to_pr(casanovo_joint_fpr, casanovo_joint_tpr)), 3)}", color='purple', lw=2)
plt.plot(*convert_to_pr(transformer_fpr, transformer_tpr), label=f"End-to-end Transformer, AUC:{round(sklearn.metrics.auc(*convert_to_pr(transformer_fpr, transformer_tpr)), 3)}", color=colors["casanovo_non_pretrained"], lw=2)
plt.plot(*convert_to_pr(binned_fpr, binned_tpr), label=f"Binned baseline, AUC:{round(sklearn.metrics.auc(*convert_to_pr(binned_fpr, binned_tpr)), 3)}0", lw=2, color=colors['binned_embeddings'])
plt.plot(*convert_to_pr(baseline_fpr, baseline_tpr), label=f"GlyCounter + XGBoost, AUC:{round(sklearn.metrics.auc(*convert_to_pr(baseline_fpr, baseline_tpr)), 3)}", color='orange', lw=2)
plt.plot(*convert_to_pr(fpr, tpr), label=f"138/144 Ratio baseline, AUC:{round(sklearn.metrics.auc(*convert_to_pr(fpr, tpr)), 3)}", color='teal', lw=2)

# Random classifier baseline
plt.plot([0, 1], [0, 1], color=colors["random"], lw=2, linestyle='--')

# Set axis labels with larger font
plt.xlabel("Recall", fontsize=10)
plt.ylabel("Precision", fontsize=10)

# Increase font size for legend
plt.legend(loc="lower left", fontsize=6)

# Despine (remove top and right borders)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("glyco_pr.svg", format="svg", bbox_inches="tight")

# Show plot
plt.show()


## PCAs of embeddings 

### Phospho

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Increase figure resolution
plt.figure(figsize=(3, 3), dpi=300)

pos = [emb.numpy() for emb in torch.load(f"phospho/embeddings_multi/phospho.pt")]
neg = [emb.numpy() for emb in torch.load(f"phospho/embeddings_multi/non_phospho.pt")]
valid_embs = pos + neg

pca = PCA(n_components=2)
pca.fit(valid_embs)
transformed_data = pca.transform(valid_embs)

# Explained variance ratio (how much variance is explained by each component)
explained_variance = pca.explained_variance_ratio_

plt.scatter(transformed_data[len(pos):,0], transformed_data[len(pos):,1], s=.1, alpha=0.2, label='Non Phospho')
plt.scatter(transformed_data[:len(pos),0], transformed_data[:len(pos),1], s=.1, alpha=0.2, label='Phosho')

plt.xticks([])
plt.yticks([])
plt.xlabel(f'PC1 ({100 * explained_variance[0]:.1f}%)', size=15)
plt.ylabel(f'PC2 ({100 * explained_variance[1]:.1f}%)', size=15)
plt.yticks([])
plt.legend(prop={'size': 7}, markerscale=8)
plt.savefig("phospho_pca_multi.png", format="png", bbox_inches="tight")
plt.show()


plt.figure(figsize=(3, 3), dpi=300)
with open(f"phospho/embeddings/1/PXD012174_combined_embeddings.pkl", "rb") as f:
    valid_embs = pickle.load(f)

with open(f"phospho/embeddings/1/PXD012174_combined_labels.pkl", "rb") as f:
    valid_labels = pickle.load(f)

pca = PCA(n_components=10)
pca.fit(valid_embs)

# Transform the data to the new principal components
transformed_data = pca.transform(valid_embs)

# Explained variance ratio (how much variance is explained by each component)
explained_variance = pca.explained_variance_ratio_

labels = [i == 0 for i in valid_labels]
inv_labels = [i == 1 for i in valid_labels]
print(len(inv_labels))
plt.scatter(transformed_data[inv_labels,0], transformed_data[inv_labels,1], s=.1, alpha=0.2, label='Non Phospho')
plt.scatter(transformed_data[labels,0], transformed_data[labels,1], s=.1, alpha=0.2, label='Phospho')

plt.xticks([])
plt.yticks([])
plt.xlabel(f'PC1 ({100 * explained_variance[0]:.1f}%)', size=15)
plt.ylabel(f'PC2 ({100 * explained_variance[1]:.1f}%)', size=15)
plt.yticks([])
plt.legend(prop={'size': 7}, markerscale=8)
plt.savefig("phospho_pca.png", format="png", bbox_inches="tight")
plt.show()



### Chimera

In [ ]:
plt.figure(figsize=(3, 3), dpi=300)

pos = [emb.numpy() for emb in torch.load(f"chimera/embeddings_multi/chimeric_test.pt")]
neg = [emb.numpy() for emb in torch.load(f"chimera/embeddings_multi/single_test.pt")]
valid_embs = pos + neg
print(len(valid_embs))

pca = PCA(n_components=2)
pca.fit(valid_embs)
transformed_data = pca.transform(valid_embs)

# Explained variance ratio (how much variance is explained by each component)
explained_variance = pca.explained_variance_ratio_
print(explained_variance)

plt.scatter(transformed_data[len(pos):,0], transformed_data[len(pos):,1], s=.1, alpha=0.15, label='Single')
plt.scatter(transformed_data[:len(pos),0], transformed_data[:len(pos),1], s=.1, alpha=0.15, label='Chimeric')

plt.xticks([])
plt.yticks([])
plt.xlabel(f'PC1 ({100 * explained_variance[0]:.1f}%)', size=15)
plt.ylabel(f'PC2 ({100 * explained_variance[1]:.1f}%)', size=15)
plt.yticks([])
plt.legend(prop={'size': 7}, markerscale=8)

plt.savefig("chimera_pca_multi.png", format="png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(3, 3), dpi=300)
pos_file = 'chimera/embeddings/test_chimeric.pt'
neg_file = 'chimera/embeddings/test_single.pt'
pos = [emb.numpy() for emb in torch.load(pos_file)]
neg = [emb.numpy() for emb in torch.load(neg_file)]
valid_embs = pos + neg
print(len(valid_embs))

pca = PCA(n_components=10)
pca.fit(valid_embs)
transformed_data = pca.transform(valid_embs)

# Explained variance ratio (how much variance is explained by each component)
explained_variance = pca.explained_variance_ratio_
print(explained_variance)

plt.scatter(transformed_data[len(pos):,0], transformed_data[len(pos):,1], s=.1, alpha=0.15, label='Single')
plt.scatter(transformed_data[:len(pos)+15000,0], transformed_data[:len(pos)+15000,1], s=.1, alpha=0.15, label='Chimeric')

plt.xticks([])
plt.yticks([])
plt.xlabel(f'PC1 ({100 * explained_variance[0]:.1f}%)', size=15)
plt.ylabel(f'PC2 ({100 * explained_variance[1]:.1f}%)', size=15)
plt.yticks([])
plt.legend(prop={'size': 7}, markerscale=8)

plt.savefig("chimera_pca.png", format="png", bbox_inches="tight")
plt.show()

### Quality

In [ ]:
plt.figure(figsize=(3, 3), dpi=300)

pos = [emb.numpy() for emb in torch.load(f"quality/embeddings_multi/test_pos.pt")]
neg = [emb.numpy() for emb in torch.load(f"quality/embeddings_multi/test_neg.pt")]
valid_embs = pos + neg
print(len(valid_embs))

pca = PCA(n_components=2)
pca.fit(valid_embs)
transformed_data = pca.transform(valid_embs)

# Explained variance ratio (how much variance is explained by each component)
explained_variance = pca.explained_variance_ratio_
print(explained_variance)

plt.scatter(transformed_data[len(pos):,0], transformed_data[len(pos):,1], s=.1, alpha=0.15, label='Low quality')
plt.scatter(transformed_data[:len(pos)-10000,0], transformed_data[:len(pos)-10000,1], s=.1, alpha=0.15, label='High quality')

plt.xticks([])
plt.yticks([])
plt.xlabel(f'PC1 ({100 * explained_variance[0]:.1f}%)', size=15)
plt.ylabel(f'PC2 ({100 * explained_variance[1]:.1f}%)', size=15)
plt.yticks([])
plt.legend(prop={'size': 7}, markerscale=8)

plt.savefig("quality_pca_multi.png", format="png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(3, 3), dpi=300)
pos_file = "quality/embeddings/test_pos.pt"
neg_file = "quality/embeddings/test_neg.pt"
pos = [emb.numpy() for emb in torch.load(pos_file)]
neg = [emb.numpy() for emb in torch.load(neg_file)]
valid_embs = pos + neg
print(len(valid_embs))

pca = PCA(n_components=2)
pca.fit(valid_embs)
transformed_data = pca.transform(valid_embs)

# Explained variance ratio (how much variance is explained by each component)
explained_variance = pca.explained_variance_ratio_
print(explained_variance)


plt.scatter(transformed_data[len(pos):,0], transformed_data[len(pos):,1], s=.1, alpha=0.15, label='Low quality')
plt.scatter(transformed_data[:len(pos)+5000,0], transformed_data[:len(pos)+5000,1], s=.1, alpha=0.15, label='High quality')
plt.xticks([])
plt.yticks([])
plt.xlabel(f'PC1 ({100 * explained_variance[0]:.1f}%)', size=15)
plt.ylabel(f'PC2 ({100 * explained_variance[1]:.1f}%)', size=15)
plt.yticks([])
plt.legend(prop={'size': 7}, markerscale=8)

plt.savefig("quality_pca.png", format="png", bbox_inches="tight")
plt.show()

## Table of phospho results

In [6]:
def calculate_metrics(predictions, y_true, threshold=0.5):
    
    # Convert probabilities to binary predictions
    y_pred = [1 if p >= threshold else 0 for p in predictions]
    
    # Compute Balanced Accuracy
    bacc = balanced_accuracy_score(y_true, y_pred)
    
    # Compute F1 Score
    f1 = f1_score(y_true, y_pred)

    # Compute ROC-AUC
    casanovo_fpr, casanovo_tpr, _ = sklearn.metrics.roc_curve(y_true, predictions)
    roc_auc = sklearn.metrics.auc(casanovo_fpr, casanovo_tpr)
    
    return bacc, f1, roc_auc

In [7]:
# Results copied from AHLF paper
cross_val_results_table= """
KOC-7C 86232 35842 0.97 0.96 0.96 0.94 0.99 0.99
SH-SY5Y 43411 53619 0.98 0.96 0.98 0.96 1.00 0.99
OVISE 83839 30848 0.97 0.95 0.96 0.93 0.99 0.99
OVAS 90936 37720 0.96 0.95 0.95 0.92 0.99 0.99
TOV-21-Primary 62350 26978 0.96 0.94 0.95 0.92 0.99 0.99
ES2-Primary 16297 6667 0.96 0.94 0.95 0.91 0.99 0.99
143B.TK 24840 158853 0.95 0.90 0.97 0.96 0.99 0.97
Primary-Ovarian 1202883 505878 0.96 0.91 0.94 0.85 0.99 0.97
Primary-BOEC 23635 61509 0.94 0.87 0.95 0.94 0.98 0.96
A673 219715 134268 0.95 0.88 0.94 0.84 0.99 0.96
THP1 18212 10353 0.87 0.89 0.84 0.85 0.96 0.96
Daudi 150915 210916 0.93 0.89 0.93 0.90 0.98 0.96
Kit255 235348 108432 0.92 0.87 0.91 0.83 0.98 0.95
HUVEC 41866 90733 0.93 0.86 0.94 0.91 0.98 0.95
U2OS 92329 205353 0.93 0.77 0.96 0.90 0.98 0.95
Primary-Glioblastoma 13318 20560 0.96 0.81 0.96 0.88 0.99 0.94
H1975 77599 54252 0.90 0.86 0.88 0.84 0.96 0.94
Primary-Glioma 974 7505 0.94 0.86 0.95 0.93 0.99 0.94
DG75 310793 260765 0.82 0.86 0.79 0.85 0.92 0.94
BT474 16717 165669 0.89 0.80 0.91 0.96 0.93 0.94
HaCaT 19216 113775 0.94 0.78 0.97 0.95 0.99 0.93
A431 354937 289691 0.88 0.85 0.87 0.83 0.97 0.93
Primary-Melanoma 17538 79643 0.89 0.84 0.90 0.91 0.96 0.93
MCF7 132269 251634 0.87 0.85 0.87 0.88 0.95 0.93
RKO 1510 30862 0.88 0.72 0.97 0.98 0.95 0.93
H358 6994 1985 0.94 0.83 0.91 0.65 0.98 0.93
SW480 228644 215187 0.80 0.84 0.76 0.84 0.91 0.93
HT-29 1625 27531 0.90 0.72 0.96 0.97 0.96 0.92
HeLa 1469194 2949614 0.89 0.83 0.89 0.89 0.95 0.92
H3255 30409 18564 0.79 0.83 0.74 0.79 0.90 0.92
COLO-205 1415 28427 0.89 0.76 0.96 0.97 0.96 0.92
HEPG2 426 45416 0.88 0.76 0.95 0.98 0.95 0.92
CACO-2 868 18490 0.89 0.76 0.96 0.97 0.96 0.92
11-18 40500 7322 0.77 0.76 0.67 0.63 0.92 0.91
Fibroblast 59624 109225 0.90 0.83 0.90 0.86 0.96 0.91
SKM-1 50816 198251 0.62 0.83 0.50 0.91 0.73 0.91
U937 111675 53302 0.55 0.83 0.23 0.76 0.63 0.91
OMP2 828 341 0.62 0.82 0.41 0.75 0.76 0.91
GB2 1819 15255 0.64 0.82 0.54 0.90 0.75 0.91
SW1398 1616 27050 0.89 0.67 0.96 0.97 0.96 0.91
DLD-1 1217 18376 0.88 0.68 0.96 0.97 0.95 0.91
A549 4068 172792 0.74 0.82 0.73 0.92 0.86 0.91
Colon 8359 28798 0.64 0.81 0.52 0.90 0.79 0.90
MV-4-11 69115 185177 0.61 0.82 0.45 0.87 0.74 0.90
SK-N-BE 3593 33997 0.91 0.77 0.93 0.93 0.97 0.90
Muscle 5257 24645 0.96 0.81 0.97 0.89 0.99 0.90
HCT116 181559 495415 0.87 0.79 0.90 0.88 0.94 0.89
P31.Fuj 2732 26885 0.60 0.80 0.41 0.90 0.71 0.89
RL 2384 559 0.61 0.79 0.36 0.66 0.76 0.89
Metastasis-Pancreas 127947 1115 0.74 0.79 0.39 0.15 0.89 0.89
U266B1 187 114 0.63 0.80 0.46 0.75 0.77 0.88
MDA-MB-231 17818 40798 0.81 0.80 0.80 0.82 0.91 0.88
Primary-Gastro 22026 219767 0.77 0.72 0.73 0.94 0.80 0.88
HCCLM6 34682 86592 0.65 0.76 0.57 0.88 0.74 0.87
Primary-Liver 7147 1239 0.68 0.78 0.47 0.48 0.81 0.87
MCF10A 27581 73430 0.89 0.78 0.91 0.80 0.97 0.87
CTS 1181 1321 0.61 0.78 0.39 0.76 0.73 0.87
LNCaP 53851 5200 0.60 0.78 0.27 0.45 0.72 0.87
RPMI-8226 1184 413 0.63 0.76 0.44 0.65 0.77 0.87
HEK293 322811 332690 0.61 0.77 0.49 0.73 0.71 0.86
Primary-Prostate 9223 100617 0.85 0.77 0.92 0.89 0.93 0.86
Primary-Pancreas 108996 44781 0.72 0.71 0.61 0.60 0.92 0.86
SU-DHL-6 2263 732 0.60 0.77 0.35 0.65 0.73 0.86
Mono-Mac-1 42898 220287 0.60 0.75 0.46 0.91 0.71 0.86
DoHH2 2664 658 0.61 0.75 0.37 0.61 0.75 0.86
Primary-Colorectal 15974 67377 0.74 0.72 0.70 0.90 0.85 0.86
Primary-Breast-MicroAndExo 6162 38360 0.63 0.75 0.58 0.78 0.72 0.85
Primary-AML 494184 5893 0.60 0.72 0.08 0.29 0.74 0.85
Kasumi-1 2294 29470 0.57 0.75 0.41 0.88 0.64 0.85
Brain 5817 26062 0.63 0.76 0.51 0.84 0.73 0.85
Jurkat 577817 1000582 0.65 0.75 0.53 0.82 0.79 0.84
ACHN 4 41 0.29 0.72 0.13 0.61 0.25 0.82
Platelets 28321 42613 0.54 0.71 0.31 0.78 0.61 0.80
BoneMarrow 241075 252733 0.59 0.67 0.45 0.74 0.65 0.80
PANC-10-05 550 1009 0.55 0.69 0.32 0.61 0.63 0.79
SW1990 661 959 0.54 0.70 0.33 0.65 0.59 0.79
PL45 483 1008 0.57 0.69 0.35 0.64 0.64 0.79
BxPC-3 905 947 0.57 0.68 0.33 0.58 0.68 0.78
HPAC 594 807 0.55 0.67 0.29 0.56 0.63 0.78
Capan-1 1079 1019 0.53 0.67 0.29 0.59 0.58 0.77
WM239A 44418 168809 0.61 0.66 0.49 0.88 0.70 0.77
SU.86.86 909 984 0.55 0.65 0.29 0.53 0.63 0.77
PANC-04-03 768 1045 0.55 0.68 0.30 0.61 0.63 0.76
CFPAC-1 999 780 0.54 0.64 0.25 0.52 0.66 0.76
HPAF-II 1115 753 0.56 0.64 0.32 0.49 0.68 0.75
PANC-05-04 1079 1426 0.56 0.65 0.33 0.55 0.65 0.74
Capan-2 539 1160 0.55 0.65 0.31 0.57 0.63 0.74
H1299 19233 23914 0.70 0.65 0.66 0.70 0.81 0.73
PANC-03-27 468 823 0.56 0.64 0.33 0.56 0.64 0.72
M019i 496964 45450 0.84 0.65 0.50 0.23 0.91 0.72
PANC-02-03 273 815 0.55 0.65 0.35 0.56 0.61 0.72
PC9 800 1627 0.55 0.64 0.26 0.76 0.68 0.72
AsPC-1 473 606 0.56 0.60 0.29 0.47 0.67 0.71
PANC-08-13 722 993 0.54 0.61 0.30 0.48 0.61 0.70
OVSAYO 11515 28 0.55 0.56 0.03 0.02 0.47 0.69
ECV-304 26897 264 0.53 0.64 0.03 0.05 0.64 0.69
Hs-700-T 716 846 0.55 0.60 0.30 0.46 0.62 0.69
Hs-766-T 515 856 0.57 0.61 0.35 0.50 0.65 0.68
Lung 348289 2974 0.56 0.63 0.04 0.04 0.63 0.68
CL1-0 13 346 0.52 0.51 0.58 0.95 0.56 0.61
HDMVEC 4320 2961 0.69 0.59 0.60 0.40 0.77 0.60
"""


In [8]:
with open("phospho/results/1/PXD012174_phospho_labels.pkl", "rb") as f:
    phospho_labels = pickle.load(f)

with open("phospho/results/1/PXD012174_casanovo_predictions.pkl", "rb") as f:
    cn_predictions = pickle.load(f)

with open("phospho/results/1/PXD012174_depthcharge_predictions.pkl", "rb") as f:
    dc_predictions = pickle.load(f)

with open(f"phospho/results/1/PXD012174_casanovo_predictions_multi.pkl", "rb") as f:
    cn_predictions_multi = pickle.load(f)



In [9]:
# Replace with predictions from model to print per-dataset results for
predictions = dc_predictions

# Define column headers with Casanovo metrics added
headers = ["Dataset", "n_non-phospho", "n_phospho", "Bacc-AHLF", "Bacc-Transformer", "F1-AHLF", "F1-Transformer", "ROC_AUC-AHLF", "ROC_AUC-Transformer"]
columns_to_keep = [0, 1, 2, 4, 6, 8]  # Indices of columns to keep from the original dataset

# Convert data into a list of lists for structured formatting
data = [ds_ln.split(' ') for ds_ln in cross_val_results_table.split("\n")[1:-1]]

# Initialize the processed data list
filtered_data = []

for row in data:
    dataset_name = row[0]  # Extract dataset name

    # Extract only the necessary columns
    selected_columns = [row[i] for i in columns_to_keep]

    # Compute Casanovo metrics if dataset is in splits['a']
    if dataset_name in predictions.keys():
        bacc_casanovo, f1_casanovo, rauc_casanovo = calculate_metrics(predictions[dataset_name], phospho_labels[dataset_name], threshold=0.5)
        selected_columns.insert(4, f"{bacc_casanovo:.3f}")  # Insert Bacc-Casanovo after Bacc-A
        selected_columns.insert(6, f"{f1_casanovo:.3f}")    # Insert F1-Casanovo after F1-A
        selected_columns.insert(8, f"{rauc_casanovo:.3f}")  # Insert RAUC-Casanovo after RAUC-A
    else:
        selected_columns.insert(4, "N/A")  # If no Casanovo metrics, fill with N/A
        selected_columns.insert(6, "N/A")
        selected_columns.insert(8, "N/A")

    filtered_data.append(selected_columns)

# Determine column widths dynamically based on the longest item in each column
col_widths = [max(len(str(item)) for item in col) for col in zip(headers, *filtered_data)]

# Print header row
header_row = "\t".join(f"{headers[i]}" for i in range(len(headers)))
print(header_row)
#print("=" * len(header_row))  # Separator line

# Print data rows
for row in filtered_data:
    if row[0] in predictions.keys():
        print("\t".join(f"{row[i]}" for i in range(len(row))))


Dataset	n_non-phospho	n_phospho	Bacc-AHLF	Bacc-Transformer	F1-AHLF	F1-Transformer	ROC_AUC-AHLF	ROC_AUC-Transformer
OVAS	90936	37720	0.95	0.978	0.92	0.967	0.99	0.997
TOV-21-Primary	62350	26978	0.94	0.980	0.92	0.967	0.99	0.997
ES2-Primary	16297	6667	0.94	0.975	0.91	0.947	0.99	0.996
Daudi	150915	210916	0.89	0.971	0.90	0.975	0.96	0.994
U2OS	92329	205353	0.77	0.961	0.90	0.978	0.95	0.992
HaCaT	19216	113775	0.78	0.969	0.95	0.986	0.93	0.993
HT-29	1625	27531	0.72	0.983	0.97	0.993	0.92	0.998
HeLa	1469194	2949614	0.83	0.959	0.89	0.973	0.92	0.990
HEPG2	426	45416	0.76	0.989	0.98	0.995	0.92	0.999
A549	4068	172792	0.82	0.961	0.92	0.981	0.91	0.991
Colon	8359	28798	0.81	0.912	0.90	0.937	0.90	0.971
Primary-Gastro	22026	219767	0.72	0.942	0.94	0.971	0.88	0.981
LNCaP	53851	5200	0.78	0.939	0.45	0.716	0.87	0.980
RPMI-8226	1184	413	0.76	0.929	0.65	0.907	0.87	0.985
HEK293	322811	332690	0.77	0.889	0.73	0.894	0.86	0.952
Primary-Prostate	9223	100617	0.77	0.960	0.89	0.979	0.86	0.990
Primary-AML	494184	5893	0.72	0.